# Campaign Targeting Decision Assistant

Marketing teams need to decide which customers to contact when campaign capacity is limited. This assistant uses existing campaign analysis to recommend a targeting strategy at a chosen audience size. It can compare up to three targeting depths and explain how the leading strategy compares with the alternatives.

The campaign analysis uses synthetic holdout data. The assistant retrieves its estimated incremental activations and uses Python to calculate comparisons.

## 1. Setup

Start Ollama with `qwen3:4b-instruct-2507-q4_K_M` available. This notebook also needs Python packages `pandas`, `ollama`, and `ipython`.

Place the campaign analysis exports `uplift_summary.csv` and `response_summary.csv` in `campaign_assistant_data` under the notebook's working directory, or change `DATA_FOLDER` in the next cell. Run the setup and function cells in order before trying a question.

Each live question is saved to `assistant_demo_runs.jsonl` with its interpretation, answer, tool results, model calls, token counts, and elapsed time.

In [1]:
import json
from pathlib import Path
from datetime import datetime
from time import perf_counter

import pandas as pd
from ollama import Client
from IPython.display import display, Markdown

# Change this path if the two CSV exports are stored elsewhere.
DATA_FOLDER = Path("campaign_assistant_data")

# Save test runs here without overwriting earlier results.
LOG_FILE = Path("assistant_demo_runs.jsonl")

# Connect to the language model running on this computer.
MODEL = "qwen3:4b-instruct-2507-q4_K_M"
client = Client(host="http://localhost:11434")

# Identify the instruction version used for each saved run.
PROMPT_VERSION = "v19"

# Load the results from the existing campaign analysis.
cumulative_summary = pd.read_csv(DATA_FOLDER / "uplift_summary.csv")
cumulative_response_summary = pd.read_csv(DATA_FOLDER / "response_summary.csv")

# Confirm that both files loaded.
print("Uplift:", cumulative_summary.shape)
print("Response:", cumulative_response_summary.shape)

Uplift: (10, 13)
Response: (10, 13)


## 2. How the strategies are compared

**Response targeting** selects customers with the highest predicted probability of activation. **Uplift targeting** selects customers with the highest predicted benefit from treatment. **Random targeting** provides a benchmark without using either score.

For response and uplift targeting, the source analysis compares the treatment and control activation rates among the selected customers to estimate incremental activations. The random benchmark scales the full-audience incremental estimate by the targeting share. This notebook compares those existing estimates at the same audience size; it does not estimate campaign outcomes itself.

## 3. Retrieve and compare the results

The comparison function looks up the estimates for one targeting depth, ranks the three strategies, and calculates the difference and percentage change for each comparison. The `compare_targeting_tool` wrapper returns the leader and its comparisons in a form the assistant can explain.

The exported analysis supports 10% through 100% of the audience in 10-percentage-point steps. Python receives each depth as a fraction: for example, 30% is `0.3`. Unsupported depths are rejected without rounding.

In [2]:
def compare_targeting_strategies(targeting_share):
    """Return strategy estimates and pairwise comparisons at one targeting depth."""

    # Only accept the targeting depths available in the exported analysis.
    available_depths = [i / 10 for i in range(1, 11)]

    if targeting_share not in available_depths:
        raise ValueError("Choose a targeting share from 0.1 to 1.0 in increments of 0.1.")

    # Retrieve the uplift and response results at the requested depth.
    uplift_row = cumulative_summary.loc[
        cumulative_summary["cumulative_targeting_share"].round(6) == targeting_share].iloc[0]

    response_row = cumulative_response_summary.loc[
        cumulative_response_summary["cumulative_targeting_share"].round(6) == targeting_share].iloc[0]

    # Combine the three strategies into one ranked table.
    results = pd.DataFrame({"strategy": ["uplift", "response", "random"],
                            "estimated_incremental_activations": [uplift_row["cumulative_incremental_activations"],
                                                                  response_row["cumulative_incremental_activations"],
                                                                  uplift_row["random_targeting_incremental_activations"]]})

    results = results.sort_values("estimated_incremental_activations", ascending=False).reset_index(drop=True)

    # Look up each strategy's estimate by name.
    activations = results.set_index("strategy")["estimated_incremental_activations"]

    # Calculate comparisons in Python so the model doesn't need to do arithmetic.
    comparisons = []

    # Create pairs of combinations for comparisons
    pairs = [("uplift", "random"),
             ("uplift", "response"),
             ("response", "random"), 
             ("response", "uplift"),
             ("random", "uplift"), 
             ("random", "response")]

    # For each pair calculate activation difference and percent change
    for strategy, baseline in pairs:
        estimate = float(activations[strategy])
        baseline_estimate = float(activations[baseline])
        
        difference = estimate - baseline_estimate

        # Percent improvement is meaningful here only with a positive baseline.
        percent_change = difference / baseline_estimate * 100 if baseline_estimate > 0 else None

        comparisons.append({"strategy": strategy,
                            "baseline_strategy": baseline,
                            "activation_difference": difference,
                            "percent_change": percent_change,
                            "targeting_share": float(targeting_share)})
        
    return results, comparisons

In [3]:
def compare_targeting_tool(targeting_share: float) -> dict:
    """Compare strategies at a targeting share from 0.1 to 1.0 in 0.1 increments."""
    results, comparisons = compare_targeting_strategies(targeting_share)

    # Results are sorted from highest to lowest.
    leader = results.iloc[0]["strategy"]
    result_records = results.round(2).to_dict("records")

    # Send only the leading strategy's comparisons.
    leader_comparisons = []
    for comparison in comparisons:
        if comparison["strategy"] == leader:
            row = comparison.copy()
            row["activation_difference"] = round(row["activation_difference"], 2)

            if row["percent_change"] is not None:
                row["percent_change"] = round(row["percent_change"], 2)

            leader_comparisons.append(row)

    return {"targeting_share": float(targeting_share),
            "metric": "estimated_incremental_activations",
            "leading_strategy": leader,
            "results": result_records,
            "comparisons": leader_comparisons}

## 4. Interpret the question and explain the results

The first model call identifies the question's intent and extracts requested targeting depths. Python checks the interpretation, returns the financial limitation or missing-depth question when appropriate, and runs one comparison for each requested depth, up to three.

When comparisons succeed, a second model call explains their results. Python adds validation errors separately, so an unsupported depth does not receive a recommendation. Each question is handled independently.

In [4]:
interpretation_instructions = """
    Interpret the user's campaign question. Do not answer it.
    
    Return one JSON object with exactly these fields:
    - "intent": either "activation" or "financial"
    - "depths": a list of numeric targeting shares

    Financial questions include questions about profit, profitability,
    or making the most money. Classify these as "financial".

    Convert percentages and written shares to decimal fractions.
    Preserve unsupported depths without rounding.
    If no depth is provided, return an empty list.
    Do not recommend a strategy or request tools.

    Extract every targeting depth explicitly supplied in the question.
    Return an empty depths list only when no targeting depth is supplied.
    Include unsupported depths; do not filter them out.

    Examples:
    "Compare 20%, 30%, and 60% targeting"
    → {"intent": "activation", "depths": [0.2, 0.3, 0.6]}

    "We can reach one quarter of our customers"
    → {"intent": "activation", "depths": [0.25]}
    """

explanation_instructions = """
Explain only the supplied successful results.
Write one short paragraph per supplied targeting depth.

For each depth, use this structure:
"At [percentage] targeting, choose [leading_strategy], with [estimate]
estimated incremental activations. Compared to [alternative], this
produces [difference] more activations ([percent]% increase). Compared
to [other alternative], this produces [difference] more activations
([percent]% increase)."

Replace every bracketed field with its corresponding supplied value.
Keep each comparison attached to its depth and named alternative.
Do not recalculate numbers or reverse comparison directions.
Do not invent results or substitute zero for missing values.
Discuss only depths present in the supplied results.

Do not add headings, bullet points, emojis, repeated recommendations,
combined percentage ranges, cross-depth summaries, or claims of
statistical significance.

Address every supplied depth and then stop.
"""

In [5]:
def ask_assistant(question, test_name="exploration"):
    """Answer a question and automatically save the run for review."""

    if not isinstance(question, str) or not question.strip():
        raise ValueError("Enter a non-empty question.")

    started = perf_counter()
    responses = []
    tool_log = []

    # Run the model to interpret the question and extract intent and depths.
    first = client.chat(
        model=MODEL,
        messages=[{"role": "system", 
                   "content": interpretation_instructions},
                    {"role": "user", 
                    "content": question}],
        format="json",
        stream=False,
        think=False,
        options={"num_predict": 500, "temperature": 0})
    
    # Add the first response to the list of responses for logging.
    responses.append(first)

    interpretation = None

    if first.done_reason == "length":
        answer = "The request reached its output limit. Please narrow the question."

    else:
        # Parse and validate the interpretation before executing anything.
        try:
            interpretation = json.loads(first.message.content)

            if not isinstance(interpretation, dict):
                raise ValueError("Expected a JSON object.")

            if set(interpretation) != {"intent", "depths"}:
                raise ValueError("Expected intent and depths.")

            # Store the intent and depths for further processing.
            intent = interpretation["intent"]
            depths = interpretation["depths"]

            # Validate the intent and depths.
            if intent not in ("activation", "financial"):
                raise ValueError("Unknown intent.")

            if not isinstance(depths, list):
                raise ValueError("Depths must be a list.")

            if any(
                isinstance(depth, bool)
                or not isinstance(depth, (int, float))
                or not float("-inf") < depth < float("inf")
                for depth in depths):
                raise ValueError("Depths must contain finite numbers.")

        except (ValueError, TypeError):
            answer = ("I couldn't interpret the request. "
                      "Please rephrase it and enter any targeting depths numerically.")

        # If the interpretation is valid, proceed to handle the question based on intent and depths.
        else:
            if intent == "financial":
                answer = ("Profit cannot be determined from activation estimates alone. "
                          "Cost and value information are required.")

            elif not depths:
                answer = "What percentage of customers would you like to target?"

            elif len(depths) > 3:
                answer = "Please request at most three targeting depths."

            else:
                # Python executes one comparison for each interpreted depth.
                for depth in depths:
                    arguments = {"targeting_share": depth}

                    try:
                        tool_result = compare_targeting_tool(**arguments)

                    except (ValueError, TypeError) as error:
                        tool_result = {"status": "error",
                            "message": str(error)}

                    tool_log.append({"tool": "compare_targeting_tool",
                                     "arguments": arguments,
                                     "result": tool_result})
                    
                # Store successful results.
                successful_results = [item for item in tool_log
                                    if item["result"].get("status") != "error"]

                # Store error messages.
                error_messages = [f"At {item['arguments']['targeting_share'] * 100:g}% targeting: "
                                f"{item['result']['message']}"
                                for item in tool_log
                                if item["result"].get("status") == "error"]

                answer_parts = []

                if successful_results:
                    # Send only successful results, without the original question.
                    final = client.chat(
                        model=MODEL,
                        messages=[{
                                "role": "system",
                                "content": explanation_instructions},
                            {"role": "user",
                             "content": json.dumps({"successful_results": successful_results})}],
                        stream=False,
                        think=False,
                        options={"num_predict": 500, "temperature": 0})
                    responses.append(final)
                    explanation = final.message.content

                    # If the final response is truncated, add a note to the explanation.
                    if final.done_reason == "length":
                        explanation += "\n[Response reached the output limit.]"

                    # Add the explanation to the final answer.
                    answer_parts.append(explanation)

                # Python adds errors without asking the model to interpret them.
                answer_parts.extend(error_messages)
                answer = "\n\n".join(answer_parts)

    # Capture the evidence needed to assess this answer.
    run = {"timestamp": datetime.now().astimezone().isoformat(),
           "test_name": test_name,
           "prompt_version": PROMPT_VERSION,
           "interpretation_instructions": interpretation_instructions,
           "interpretation_raw": first.message.content,
           "interpretation": interpretation,
           "explanation_instructions": explanation_instructions,
           "model": MODEL,
           "question": question,
           "answer": answer,
           "tool_log": tool_log,
           "model_calls": len(responses),
           "input_tokens": sum(response.prompt_eval_count or 0 for response in responses),
           "output_tokens": sum(response.eval_count or 0 for response in responses),
           "stop_reasons": [response.done_reason for response in responses],
           "elapsed_seconds": round(perf_counter() - started, 2)}

    # Append the run to disk so later questions cannot overwrite it.
    with LOG_FILE.open("a", encoding="utf-8") as file:
        file.write(json.dumps(run, ensure_ascii=False, allow_nan=False) + "\n")

    return run

## 5. Reviewed examples

These answers come from the recorded v19 evaluation. The numerical results were reviewed against the saved comparison outputs.

### Compare 30% and 60%

**Question:** Compare 30% and 60% targeting. Recommend a strategy at each depth and explain its advantage over the alternatives.

At 30% targeting, choose uplift, with 274.22 estimated incremental activations. Compared to random, this produces 54.33 more activations (24.71% increase). Compared to response, this produces 91.84 more activations (50.36% increase).

At 60% targeting, choose response, with 478.97 estimated incremental activations. Compared to random, this produces 39.19 more activations (8.91% increase). Compared to uplift, this produces 28.52 more activations (6.33% increase).

### When random targeting leads

**Question:** We can contact 40% of customers. Which strategy should we use and why?

At 40% targeting, choose random, with 293.19 estimated incremental activations. Compared to uplift, this produces 6.76 more activations (2.36% increase). Compared to response, this produces 34.5 more activations (13.34% increase).

### Supported and unsupported depths

**Question:** Compare targeting at 25%, thirty percent, and .5. Recommend a strategy at each depth.

At 30% targeting, choose uplift, with 274.22 estimated incremental activations. Compared to random, this produces 54.33 more activations (24.71% increase). Compared to response, this produces 91.84 more activations (50.36% increase).

At 50% targeting, choose uplift, with 381.7 estimated incremental activations. Compared to random, this produces 15.22 more activations (4.15% increase). Compared to response, this produces 12.82 more activations (3.48% increase).

At 25% targeting: Choose a targeting share from 0.1 to 1.0 in increments of 0.1.

## 6. Evaluation and design decisions

The development notebook documents 19 versions and the tests used to refine the assistant. Early runs exposed incorrect or mixed numerical comparisons, omitted depths, inconsistent handling of written shares, and activation recommendations in response to profit questions.

The final workflow separates question interpretation, Python execution, and explanation. Python handles unsupported-depth errors separately after earlier model explanations invented results for an error.

All 17 questions passed the v19 evaluation with the 4B model. The suite covered single and multiple depths, written shares, unsupported and missing depths, and financial questions. A separate four-depth boundary check also passed. The application can compare estimated activations, but it cannot determine profit without cost and value information.

The larger 9B model handled written-depth examples in earlier testing but took longer. The final workflow and evaluation used the 4B model; the 9B model was not retested with that workflow.

## 7. Try a question

Run the cell below and enter your question when prompted. For example: `Compare 30% and 60% targeting` or `Which strategy should I use for half of the customers?`

Each question is independent, so include the audience percentage in each question. The answer and its comparison evidence are displayed together. Run the cell again to ask another question.

In [7]:
question = input("Ask about a targeting decision: ")
if question.strip():
    result = ask_assistant(question, test_name="demo_question")
    display(Markdown(result["answer"]))
    print("Tool evidence:")
    print(json.dumps(result["tool_log"], indent=2))
    print(f"Model calls: {result['model_calls']} | Response time: {result['elapsed_seconds']} seconds")
else:
    print("Enter a question and run this cell again.")

At 30% targeting, choose uplift, with 274.22 estimated incremental activations. Compared to random, this produces 54.33 more activations (24.71% increase). Compared to response, this produces 91.84 more activations (50.36% increase).

At 60% targeting, choose response, with 478.97 estimated incremental activations. Compared to random, this produces 39.19 more activations (8.91% increase). Compared to uplift, this produces 28.52 more activations (6.33% increase).

At 35% targeting: Choose a targeting share from 0.1 to 1.0 in increments of 0.1.

Tool evidence:
[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.84,
          "percent_change": 50.36,
          "t